In [ ]:
%load_ext autoreload
%autoreload 2
from omegaconf import OmegaConf
from oat.data_module import factory
from oat.model_module import model
import torch
from oat.data_module.cosmo_peakweather_dataset import CosmoPeakWeatherDataset
from oat.data_module import data_split
from torch.utils.data import DataLoader

In [ ]:
from hydra import compose, initialize
import torch

with initialize(config_path="../configs/", version_base=None):
    cfg = compose(config_name="config")

dataset = CosmoPeakWeatherDataset(cfg.dataset)
train_data, val_data, test_data = data_split.create_splits(
    dataset, **cfg.split)
train_loader = DataLoader(train_data, batch_size=2, shuffle=True)

In [ ]:
model_cfg = cfg.model
data_cfg = cfg.dataset

global_input_chans = len(data_cfg.nwp_vars) + len(data_cfg.topo_vars)
global_encoder_cfg = model.GlobalEncoderConfig(
    **model_cfg.global_encoder, 
    in_channels=global_input_chans,
    field_extent_max=dataset.nwp_extent['max'],
    field_extent_min=dataset.nwp_extent['min']
    )
topography_encoder_cfg = model.TopographicEncoderConfig(
    **model_cfg.topographic_encoder,
    in_channels=len(data_cfg.topo_vars)
    )
fusion_layer_cfg = model.FusionLayerConfig(
    **model_cfg.fusion_layer,
    out_channels=model_cfg.d_model)
masking_cfg = model.MaskingConfig(**model_cfg.masking)
transformer_cfg = model.TransformerConfig(
    d_geo_bias=len(data_cfg.geo_vars),
    d_model=model_cfg.d_model,
    **model_cfg.transformer)
output_head_cfg = model.OutputHeadConfig(
    target_features=len(data_cfg.obs_vars),
    in_features=model_cfg.d_model,
    **model_cfg.output_head)
oat_config = model.OATConfig(
    global_encoder=global_encoder_cfg,
    topographic_encoder=topography_encoder_cfg,
    fusion_layer=fusion_layer_cfg,
    masking=masking_cfg,
    transformer=transformer_cfg,
    output_head=output_head_cfg,
    num_samples=model_cfg.num_samples
)


In [ ]:
oat_model = model.OAT(oat_config)

In [ ]:
batch = next(iter(train_loader))

output = oat_model(global_data=batch['global_data'], 
                   observation_data=batch['observation_data'], 
                   query_data=batch['query_data'],)

In [ ]:
batch = next(iter(loader))

In [ ]:
obs_data = batch['observation_data']
num_stations = obs_data['obs'].shape[1]
key_stations = 100
query = {
    'coords': obs_data['coords'][:, key_stations:, :],
    'topo': obs_data['topo'][:, key_stations:, :],
    'geo': obs_data['geo'][:, key_stations:, :],
    'obs': obs_data['obs'][:, key_stations:, :],
    'mask_obs': obs_data['mask_obs'][:, key_stations:, :],
}
observation_data = {
    'coords': obs_data['coords'][:, :key_stations, :],
    'topo': obs_data['topo'][:, :key_stations, :],
    'geo': obs_data['geo'][:, :key_stations, :],
    'obs': obs_data['obs'][:, :key_stations, :],
    'mask_obs': obs_data['mask_obs'][:, :key_stations, :],
}
batch['observation_data'] = observation_data
batch['query_data'] = query

In [ ]:
model = OAT()

In [ ]:
with torch.no_grad():
    output = model(global_data=batch['global_data'], observation_data=batch['observation_data'], query_data=batch['query_data'], )

In [ ]:
output.keys()

In [ ]:

#with torch.no_grad():
# get gradients
pred, loss = model.forward_with_loss(global_data=batch['global_data'], observation_data=batch['observation_data'], query_data=batch['query_data'], target_obs=batch['query_data']['obs'], target_mask_obs=batch['query_data']['mask_obs'])
loss.backward()

In [ ]:
pred.keys()

In [ ]:
for param in model.parameters():
    if param.grad is not None:
        print(param.grad.abs().sum())

In [ ]:
output

In [ ]:
torch.sum(batch['query_data']['obs'] == torch.nan)

In [ ]:
torch.sum(torch.any(pred['val_pred'] == torch.nan))

In [ ]:
# check if nans in output
torch.sum(output['val_pred'] == torch.nan)

In [ ]:
torch.arange(10)[]

In [ ]:
batch['observation_data']['mask_obs'][2]